# 🐱🐶 Project 1: Cat vs Dog Classification

---

## 🧠 1️⃣ Project Overview

Task: Classify images as cat or dog  
Dataset: Kaggle “Dogs vs Cats” (~25k images)  
Input: Color images (RGB), usually resized to 128x128 or 150x150  
Output: 0 = Cat, 1 = Dog  
Challenge: Large dataset → need data augmentation to reduce overfitting  

---

## 🔹 2️⃣ Preprocessing Images

Keras ImageDataGenerator


In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Training data augmentation
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    validation_split=0.2
)

train_generator = train_datagen.flow_from_directory(
    'data/cats_dogs',
    target_size=(128,128),
    batch_size=32,
    class_mode='binary',
    subset='training'
)

validation_generator = train_datagen.flow_from_directory(
    'data/cats_dogs',
    target_size=(128,128),
    batch_size=32,
    class_mode='binary',
    subset='validation'
)


✅ Notes

- rescale=1./255 → normalize pixels  
- rotation_range, zoom_range → helps model generalize  
- subset='training' & subset='validation' → split dataset automatically  

---

## 🟢 3️⃣ CNN Model for Cat vs Dog


In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout

model = Sequential(
    [
        Conv2D(32, (3, 3), activation="relu", input_shape=(128, 128, 3)),
        MaxPooling2D(2, 2),
        Conv2D(64, (3, 3), activation="relu"),
        MaxPooling2D(2, 2),
        Conv2D(128, (3, 3), activation="relu"),
        MaxPooling2D(2, 2),
        Flatten(),
        Dense(128, activation="relu"),
        Dropout(0.5),
        Dense(1, activation="sigmoid"),  # Binary output
    ]
)

model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
model.summary()

## 🔵 4️⃣ Training the Model

In [ ]:
history = model.fit(
    train_generator,
    steps_per_epoch=train_generator.samples // train_generator.batch_size,
    validation_data=validation_generator,
    validation_steps=validation_generator.samples // validation_generator.batch_size,
    epochs=10,
)

✅ Notes

- Dropout(0.5) → prevents overfitting  
- steps_per_epoch → total training images / batch size  

---

## 🟣 5️⃣ Visualize Training Accuracy & Loss


In [ ]:
import matplotlib.pyplot as plt

plt.plot(history.history["accuracy"], label="train acc")
plt.plot(history.history["val_accuracy"], label="val acc")
plt.plot(history.history["loss"], label="train loss")
plt.plot(history.history["val_loss"], label="val loss")
plt.legend()
plt.show()

- Helps check overfitting

- If validation drops → consider more augmentation or dropout

## 🔹 6️⃣ Predicting New Images

In [ ]:
from tensorflow.keras.preprocessing import image
import numpy as np

img = image.load_img("data/cats_dogs/test/cat1.jpg", target_size=(128, 128))
x = image.img_to_array(img)
x = np.expand_dims(x, axis=0) / 255.0

prediction = model.predict(x)
print("Dog" if prediction[0][0] > 0.5 else "Cat")

✅ Simple real-time inference on a single image.

## 🔹 7️⃣ Advanced Tips for Cat vs Dog

- Use Pretrained Models (Transfer Learning) → VGG16, ResNet50 for better accuracy

- Fine-tune last layers → improve generalization

- Use EarlyStopping to stop training when validation stops improving

# Full Project

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
import matplotlib.pyplot as plt
from tensorflow.keras.preprocessing import image
import numpy as np

# Training data augmentation
train_datagen = ImageDataGenerator(
    rescale=1.0 / 255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    validation_split=0.2,
)

train_generator = train_datagen.flow_from_directory(
    "data/cats_dogs",
    target_size=(128, 128),
    batch_size=32,
    class_mode="binary",
    subset="training",
)

validation_generator = train_datagen.flow_from_directory(
    "data/cats_dogs",
    target_size=(128, 128),
    batch_size=32,
    class_mode="binary",
    subset="validation",
)


model = Sequential(
    [
        Conv2D(32, (3, 3), activation="relu", input_shape=(128, 128, 3)),
        MaxPooling2D(2, 2),
        Conv2D(64, (3, 3), activation="relu"),
        MaxPooling2D(2, 2),
        Conv2D(128, (3, 3), activation="relu"),
        MaxPooling2D(2, 2),
        Flatten(),
        Dense(128, activation="relu"),
        Dropout(0.5),
        Dense(1, activation="sigmoid"),  # Binary output
    ]
)

model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
model.summary()


history = model.fit(
    train_generator,
    steps_per_epoch=train_generator.samples // train_generator.batch_size,
    validation_data=validation_generator,
    validation_steps=validation_generator.samples // validation_generator.batch_size,
    epochs=10,
)


plt.plot(history.history["accuracy"], label="train acc")
plt.plot(history.history["val_accuracy"], label="val acc")
plt.plot(history.history["loss"], label="train loss")
plt.plot(history.history["val_loss"], label="val loss")
plt.legend()
plt.show()


img = image.load_img("data/cats_dogs/test/cat1.jpg", target_size=(128, 128))
x = image.img_to_array(img)
x = np.expand_dims(x, axis=0) / 255.0

prediction = model.predict(x)
print("Dog" if prediction[0][0] > 0.5 else "Cat")